In [37]:
import os

In [38]:
%pwd

'C:\\Users\\Dell\\End-to-End-Machine-Learning-Project'

In [39]:

os.chdir(r"C:\Users\Dell\End-to-End-Machine-Learning-Project")

In [40]:
%pwd

'C:\\Users\\Dell\\End-to-End-Machine-Learning-Project'

In [41]:



os.environ["MLFLOW_TRACKING_URI"] = "https://dagshub.com/manas-beep/End-to-End-Machine-Learning-Project.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"] = "manas-beep"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "8d1b8587ce4ef743178fccaf9a30468a02801625"


In [42]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    all_params: dict
    metric_file_name: Path
    target_column: str
    mlflow_uri: str

In [43]:
from mlProject.constants import *
from mlProject.utils.common import read_yaml, create_directories, save_json

In [44]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH,
        schema_filepath=SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])


    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation
        params = self.params.ElasticNet
        schema = self.schema.TARGET_COLUMN

        create_directories([config.root_dir])

        model_evaluation_config = ModelEvaluationConfig(
            root_dir=config.root_dir,
            test_data_path=config.test_data_path,
            model_path=config.model_path,
            all_params=params,
            metric_file_name=config.metric_file_name,
            target_column=schema.name,
            mlflow_uri="https://dagshub.com/manas-beep/End-to-End-Machine-Learning-Project.mlflow",)

        return model_evaluation_config

In [45]:
import os
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from urllib.parse import urlparse
import mlflow
import mlflow.sklearn
import numpy as np
import joblib

In [46]:
class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config

    def eval_metrics(self, actual, pred):
        rmse = np.sqrt(mean_squared_error(actual, pred))
        mae = mean_absolute_error(actual, pred)
        r2 = r2_score(actual, pred)
        return rmse, mae, r2
    

    def log_into_mlflow(self):

        test_data = pd.read_csv(self.config.test_data_path)
        model = joblib.load(self.config.model_path)

        test_x = test_data.drop([self.config.target_column], axis=1)
        test_y = test_data[[self.config.target_column]]

        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme

        with mlflow.start_run():

            predicted_qualities = model.predict(test_x)

            (rmse, mae, r2) = self.eval_metrics(test_y, predicted_qualities)

             # Saving metrics as local
            scores = {"rmse": rmse, "mae": mae, "r2": r2}
            save_json(path=Path(self.config.metric_file_name), data=scores)

            mlflow.log_params(self.config.all_params)

            mlflow.log_metric("rmse", rmse)
            mlflow.log_metric("r2", r2)
            mlflow.log_metric("mae", mae)

            
            if tracking_url_type_store != "file":

                mlflow.sklearn.log_model( model,"model",registered_model_name="ElasticnetModel")

            else:
                mlflow.sklearn.log_model(model, "model")

In [47]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation = ModelEvaluation(config=model_evaluation_config)
    model_evaluation.log_into_mlflow()
except Exception as e:
        raise e

[2026-04-28 21:27:18,223: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-04-28 21:27:18,240: INFO: common: yaml file: params.yaml loaded successfully]
[2026-04-28 21:27:18,248: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-04-28 21:27:18,255: INFO: common: created directory at: artifacts]
[2026-04-28 21:27:18,263: INFO: common: created directory at: artifacts/model_evaluation]


[2026-04-28 21:27:21,455: INFO: common: json file saved at: artifacts\model_evaluation\metrics.json]


c:\Users\Dell\anaconda3\envs\mlproject\lib\site-packages\_distutils_hack\__init__.py:31: UserWarning: Setuptools is replacing distutils. Support for replacing an already imported distutils is deprecated. In the future, this condition will fail. Register concerns at https://github.com/pypa/setuptools/issues/new?template=distutils-deprecation.yml
  warnings.warn(
Successfully registered model 'ElasticnetModel'.
2026/04/28 21:27:40 INFO mlflow.tracking._model_registry.client: Waiting up to 300 seconds for model version to finish creation.                     Model name: ElasticnetModel, version 1
Created version '1' of model 'ElasticnetModel'.
